# Ajuste final y exportación de DART + CRNN

Este notebook entrena ambas ramas desde cero con las 100 personas y las 197 noches de Sleep-Cassette y Sleep-Telemetry. Reutiliza las características guardadas, exporta los pesos, mide el tiempo de cada ajuste y comprueba la inferencia en un EDF. Las métricas de prueba se consultan en [el notebook de evaluación 80/20](dart_crnn_mixto.ipynb).

La CRNN tiene configuradas las 27 épocas elegidas en el experimento de selección con la misma partición 80/20. DART conserva 667 iteraciones como referencia mientras se completa la comparación de iteraciones. Este notebook está pendiente de ejecución.

In [ ]:
import hashlib
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
from uuid import uuid4

os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '5')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import lightgbm as lgb
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.fft import rfft
from scipy.ndimage import convolve1d
from scipy.special import softmax
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data.dvc').is_file())
SRC = ROOT / 'packages/sleep-staging/src'
sys.path.insert(0, str(SRC))
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_cassette_records, discover_sleep_telemetry_records
from sleep_staging.training import create_lightgbm_classifier, load_or_build_supervised_dataset

inicio_ejecucion = perf_counter()
DATA = ROOT / 'data'
STORE = DATA / 'models/dart_crnn_mixto'
SPEC = json.loads((DATA / 'experimentos/lightgbm_st_sc/experimento.json').read_text('utf8'))
CLASES = np.array(SPEC['classes'])
COLUMNAS = SPEC['feature_columns']
PERFILES = {c: SPEC['preprocessing_' + c.lower()] for c in ('Telemetry', 'Cassette')}
CANALES = PERFILES['Telemetry']['channels']
ITERACIONES_DART = 667
PARAMS = dict(SPEC['parameters'], n_jobs=6, n_estimators=ITERACIONES_DART)
BANDAS = [(0.3, 4), (4, 8), (8, 12), (12, 16), (16, 30), (30, 35)]
DESFASES = (0, 5, 10)
SEMILLA = 42
EPOCAS_CRNN = 27
tf.config.experimental.enable_op_determinism()

## Datos para el ajuste final

Se incluyen las 153 noches de Cassette y las 44 de Telemetry. Las características tabulares y espectrales se leen de las cachés de data/processed.

In [ ]:
registros = {'Telemetry': discover_sleep_telemetry_records(DATA/'sleep-telemetry'),
             'Cassette': discover_sleep_cassette_records(DATA/'sleep-cassette')}
esperados = {'Cassette': (78, 153), 'Telemetry': (22, 44)}
for cohorte, lista in registros.items():
    assert (len({r.subject_id for r in lista}), len(lista)) == esperados[cohorte], cohorte
indices, tablas = [], []
for cohorte, lista in registros.items():
    pipeline = PreprocessingPipeline(PreprocessingConfig(**PERFILES[cohorte]))
    grupos = [lista] if cohorte == 'Telemetry' else [[r] for r in lista]
    cache = DATA/'processed/supervised'
    for posicion, grupo in enumerate(grupos, 1):
        ds = load_or_build_supervised_dataset(grupo, pipeline, cache, verbose=False)
        tablas.append(ds.features.reindex(columns=COLUMNAS).astype('float32'))
        indices.append(ds.metadata[['record_id','subject_id','epoch_index']].assign(cohorte=cohorte, etapa=ds.labels.to_numpy()))
        if posicion % 10 == 0 or posicion == len(grupos):
            print(f'{cohorte} · {posicion}/{len(grupos)} grupos procesados', flush=True)
X = pd.concat(tablas, ignore_index=True)
meta = pd.concat(indices, ignore_index=True)
del tablas, indices, ds
assert len(meta)==len(X) and not meta.duplicated(['record_id','epoch_index']).any()
assert sorted(meta.etapa.unique())==CLASES.tolist()
assert meta.subject_id.nunique() == 100 and meta.record_id.nunique() == 197
assert set(meta.record_id) == {r.record_id for lista in registros.values() for r in lista}
display(meta.groupby('cohorte').agg(personas=('subject_id','nunique'), noches=('record_id','nunique'),
                                    segmentos=('etapa','size')))

## DART

El paquete del proyecto prepara las características tabulares y construye el clasificador. La configuración mixta conserva el orden de sus 2.280 columnas y sus pesos por etapa. Las características de EMG no disponibles en Cassette se representan como valores ausentes.

In [ ]:
def entrenar_dart(filas):
    modelo = create_lightgbm_classifier(**PARAMS)
    def progreso(env):
        iteracion = env.iteration + 1
        if iteracion % 50 == 0 or iteracion == env.end_iteration:
            print(f'DART · {iteracion}/{env.end_iteration} iteraciones', flush=True)

    datos = X if filas.index.equals(X.index) else X.loc[filas.index]
    modelo.fit(datos, filas.etapa, callbacks=[progreso])
    np.testing.assert_array_equal(modelo.classes_, CLASES)
    return modelo.booster_

## CRNN contextual

La rama neuronal usa dos EEG, EOG y EMG. Después del filtro de 0,3–35 Hz y la normalización por época, cada segmento produce seis bloques de cinco segundos y seis bandas por canal. Se añaden la escala robusta de la noche y dos ventanas de contexto (7,5 y 2 minutos). La red combina convolución y GRU bidireccional sobre secuencias de 15 épocas. Las etiquetas de relleno y las épocas sin anotación se excluyen de la pérdida.

In [ ]:
def entrada_crnn(archivo_edf):
    with mne.io.read_raw_edf(archivo_edf, preload=True, verbose='ERROR') as raw:
        if raw.info['sfreq'] != 100:
            raise ValueError('La CRNN requiere una señal a 100 Hz.')
        presentes = [c for c in CANALES if c in raw.ch_names]
        if not set(CANALES[:3]).issubset(presentes):
            raise ValueError(f'Faltan señales EEG/EOG en {archivo_edf.name}')
        raw.pick(presentes).filter(.3, 35., verbose=False)
        n = raw.n_times // 3000
        signal = np.zeros((n, 3000, 4), dtype='float32')
        for j, ch in enumerate(CANALES):
            if ch in presentes:
                v = raw.get_data(picks=[ch], stop=n*3000).reshape(n, 3000)*1e6
                signal[:,:,j] = (v-v.mean(1,keepdims=True))/(v.std(1,keepdims=True)+1e-8)
    f = np.fft.rfftfreq(500, 1/100)
    ventana = np.hanning(500).astype('float32')[None,None,:,None]
    x = np.empty((n, 6, 24), dtype='float32')
    for i in range(0,n,100):
        bloque = signal[i:i+100].reshape(-1,6,500,4)
        potencia = np.abs(rfft(bloque*ventana, axis=2))**2/500
        x[i:i+len(bloque)] = np.concatenate([
            np.log1p(potencia[:,:,(f>=a)&(f<b)].mean(axis=2)) for a,b in BANDAS], axis=-1)
    q10, mediana, q90 = np.percentile(x.reshape(-1,24), [10,50,90], axis=0)
    z = np.clip((x-mediana)/np.maximum(q90-q10,.05),-5,5).astype('float32')
    media = z.mean(axis=1)
    triangular = np.r_[1:9,7:0:-1].astype('float32')
    centro = convolve1d(media, triangular/triangular.sum(), axis=0, mode='nearest')
    pad = np.pad(media,((3,0),(0,0)),mode='edge')
    pasado = sum(pad[i:i+n] for i in range(4))/4
    return np.concatenate([x,z,np.broadcast_to(centro[:,None],x.shape),
                           np.broadcast_to(pasado[:,None],x.shape)],axis=-1)

def bloques(x, desfase, relleno=0):
    pad = [(desfase,-(len(x)+desfase)%15)] + [(0,0)]*(x.ndim-1)
    return np.pad(x,pad,constant_values=relleno).reshape(-1,15,*x.shape[1:])

In [ ]:
def entrenar_crnn(filas, entradas):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEMILLA)
    rng = np.random.default_rng(SEMILLA)
    l = tf.keras.layers
    entrada = l.Input((15,6,96))
    z = l.TimeDistributed(l.Conv1D(48,3,padding='same',activation='relu'))(entrada)
    z = l.TimeDistributed(l.Flatten())(z)
    z = l.Dropout(.15)(l.Dense(48,activation='relu')(z))
    z = l.Bidirectional(l.GRU(32,return_sequences=True))(z)
    red = tf.keras.Model(entrada,l.Dense(5,activation='softmax')(l.Dropout(.20)(z)))
    red.compile(optimizer=tf.keras.optimizers.Adam(.001), loss='sparse_categorical_crossentropy')
    conteos = filas.etapa.value_counts().reindex(CLASES).to_numpy()
    pesos = (conteos.mean()/conteos)**.35
    pesos /= np.average(pesos,weights=conteos)
    lotes = {}
    for d in DESFASES:
        xs, ys = [], []
        for rid,g in filas.groupby('record_id'):
            y = np.full(len(entradas[rid]),-1,dtype='int32')
            y[g.epoch_index] = pd.Categorical(g.etapa,categories=CLASES).codes
            y = bloques(y,d,-1)
            valido = (y>=0).any(1)
            xs.append(bloques(entradas[rid],d)[valido])
            ys.append(y[valido])
        x,y = np.concatenate(xs),np.concatenate(ys)
        lotes[d] = (x,y.clip(0),np.where(y>=0,pesos[y.clip(0)],0).astype('float32'))
    for epoca in range(EPOCAS_CRNN):
        x,y,w = lotes[DESFASES[epoca%3]]
        orden = rng.permutation(len(x))
        red.fit(x[orden], y[orden], sample_weight=w[orden], batch_size=64, epochs=1, shuffle=False, verbose=0)
        if (epoca + 1) % 5 == 0 or epoca + 1 == EPOCAS_CRNN:
            print(f'CRNN · {epoca + 1}/{EPOCAS_CRNN} épocas', flush=True)
    return red

def prob_crnn(red,x):
    vistas = []
    for d in DESFASES:
        v = red.predict(bloques(x,d),batch_size=128,verbose=0).reshape(-1,len(CLASES))
        vistas.append(v[d:d+len(x)].astype('float64'))
    return softmax(np.log(np.maximum(vistas,1e-10)).mean(axis=0),axis=1)

## Fusión y persistencia

La media geométrica ponderada asigna un 70 % a DART y un 30 % a la CRNN. Cada fila conserva el orden N1, N2, N3, REM y W. La etapa estimada es la de mayor probabilidad conjunta. El manifiesto guarda las personas y noches de entrenamiento, la receta, las sumas SHA-256 de los pesos y los tiempos de ajuste.

In [ ]:
def combinar(p_dart, p_crnn):
    if p_dart.shape != p_crnn.shape:
        raise ValueError('Las probabilidades de ambas ramas deben estar alineadas.')
    return softmax(.70*np.log(np.maximum(p_dart,1e-7)) + .30*np.log(np.maximum(p_crnn,1e-7)), axis=1)

def sha256(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest() if hasattr(hashlib,'file_digest') else digest_stream(f)

def digest_stream(f):
    h = hashlib.sha256()
    for chunk in iter(lambda: f.read(2**20), b''):
        h.update(chunk)
    return h.hexdigest()

import inspect

def carpeta_espectral():
    receta_entrada = {
        'codigo': inspect.getsource(entrada_crnn),
        'canales': list(CANALES), 'bandas': BANDAS,
        'versiones': {p: version(p) for p in ('mne', 'numpy', 'scipy')},
    }
    contenido = json.dumps(receta_entrada, sort_keys=True, ensure_ascii=False)
    clave = hashlib.sha256(contenido.encode('utf8')).hexdigest()[:16]
    carpeta = DATA/'processed'/'crnn_espectral'/clave
    carpeta.mkdir(parents=True, exist_ok=True)
    receta_path = carpeta/'preprocesamiento.json'
    if not receta_path.exists():
        temp = carpeta/(uuid4().hex+'.tmp')
        temp.write_text(contenido, encoding='utf8')
        temp.replace(receta_path)
    return carpeta

def entrada_espectral_guardada(archivo, huella, carpeta, calculada=None):
    ruta = carpeta/f'{archivo.stem}-{huella}.npy'
    if not ruta.is_file():
        valores = entrada_crnn(archivo) if calculada is None else calculada
        temporal = carpeta/(uuid4().hex+'.npy')
        try:
            np.save(temporal, valores, allow_pickle=False)
            temporal.replace(ruta)
        finally:
            temporal.unlink(missing_ok=True)
    resultado = np.load(ruta, mmap_mode='r', allow_pickle=False)
    if resultado.dtype != np.float32 or resultado.ndim != 3 or resultado.shape[1:] != (6, 96):
        raise ValueError(f'Entrada espectral incompatible en {ruta}')
    return resultado

def carpeta_dart_inferencia(perfil, firma_paquete):
    receta = {
        'perfil': perfil, 'paquete': firma_paquete, 'formato': 'float32-epocas-completas-v1',
        'versiones': {p: version(p) for p in ('mne', 'numpy', 'scipy', 'pandas')},
    }
    contenido = json.dumps(receta, sort_keys=True, ensure_ascii=False)
    clave = hashlib.sha256(contenido.encode('utf8')).hexdigest()[:16]
    carpeta = DATA/'processed'/'dart_inferencia'/clave
    carpeta.mkdir(parents=True, exist_ok=True)
    descripcion = carpeta/'preprocesamiento.json'
    if not descripcion.exists():
        temporal = carpeta/(uuid4().hex+'.tmp')
        temporal.write_text(contenido, encoding='utf8')
        temporal.replace(descripcion)
    return carpeta

def guardar_tabla_dart(features, metadata, ruta):
    tabla = features.astype('float32').reset_index(drop=True)
    tabla.insert(0, 'onset_seconds', metadata.onset_seconds.to_numpy())
    tabla.insert(0, 'epoch_index', metadata.epoch_index.to_numpy())
    temporal = ruta.parent/(uuid4().hex+'.parquet')
    try:
        tabla.to_parquet(temporal, index=False)
        temporal.replace(ruta)
    finally:
        temporal.unlink(missing_ok=True)
    return tabla

def tabla_dart_guardada(archivo, perfil, carpeta, huella):
    ruta = carpeta/f'{archivo.stem}-{huella}.parquet'
    if ruta.is_file():
        return pd.read_parquet(ruta)
    salida = PreprocessingPipeline(PreprocessingConfig(**perfil)).transform_edf(archivo)
    return guardar_tabla_dart(salida.features, salida.metadata, ruta)

registros_planos = [r for lista in registros.values() for r in lista]
huellas = {r.record_id: {'persona':r.subject_id,'psg':sha256(r.psg_path),'hipnograma':sha256(r.hypnogram_path)}
           for r in registros_planos}
cache_espectral = carpeta_espectral()
entradas = {r.record_id: entrada_espectral_guardada(r.psg_path, huellas[r.record_id]['psg'], cache_espectral)
            for r in registros_planos}

In [ ]:
def huella_json(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

codigo = json.loads((ROOT/'Notebooks/dart_crnn_mixto_final_100.ipynb').read_text('utf8'))['cells']
fuentes = {c['id']:c['source'] for c in codigo if c['id'] in ('setup','datos','dart','espectro','crnn','fusion')}
receta = huella_json(fuentes)
paquete = huella_json({p.relative_to(SRC).as_posix():hashlib.sha256(p.read_bytes().replace(b'\r\n',b'\n')).hexdigest() for p in sorted((SRC/'sleep_staging').rglob('*.py'))})
versiones = {p:version(p) for p in ('lightgbm','tensorflow','keras','mne','numpy','scipy','scikit-learn','pandas')}

prepro_crnn = {'alcance': 'Una noche completa. Salida por época de 30 s desde el inicio del EDF. Muestras finales incompletas descartadas.',
 'frecuencia_hz': 100,
 'duracion_epoca_s': 30,
 'segmento_espectral_s': 5,
 'filtro': {'tipo': 'MNE FIR por canal', 'fmin_hz': 0.3, 'fmax_hz': 35, 'fase': 'zero', 'diseno': 'firwin'},
 'normalizacion_epoca': 'Convertir a microvoltios; restar media y dividir por desviación poblacional + 1e-8 por canal y época.',
 'canales_ausentes': 'EMG ausente se rellena con ceros después de normalizar. Se requieren ambos EEG y EOG.',
 'espectro': 'Hann simétrica 500 muestras, rFFT, potencia = abs(FFT)^2 / 500. Para cada banda [a,b), log1p(media de potencia). Orden '
             'banda, luego canal.',
 'contexto': 'Concatenar 24 valores originales, 24 robustos por noche, 24 medias centradas y 24 medias del pasado.',
 'escala_robusta': 'Mediana y percentiles 10 y 90 de todos los bloques de 5 s de la noche. Denominador máximo entre P90-P10 y 0,05. '
                   'Recorte [-5,5].',
 'ventana_centrada': 'Media de 6 bloques por época, luego 15 épocas con pesos triangulares 1..8..1; extender extremos replicando la '
                     'época del borde.',
 'ventana_pasada': 'Media de la época actual y las tres anteriores de las medias robustas por época, con replicación al inicio.',
 'secuencias': '15 épocas con padding de ceros al inicio según desfase y hasta completar el último bloque. Agregación por noche.',
 'agregacion': 'Media geométrica normalizada entre las tres posiciones de cada época, piso 1e-10 por probabilidad de vista.'}

def contrato(filas):
    return {'receta':receta, 'paquete':paquete, 'versiones':versiones,
            'registros_entrenamiento':{r:huellas[r] for r in sorted(filas.record_id.unique())},
            'personas_entrenamiento':sorted(filas.subject_id.unique()),'segmentos_entrenamiento':len(filas),
            'clases':CLASES.tolist(),'columnas_dart':COLUMNAS,'parametros_dart':PARAMS,
            'preprocesamiento_crnn':prepro_crnn, 'preprocesamiento_dart':PERFILES, 'canales_crnn':CANALES, 'bandas_hz':BANDAS,
            'epocas_crnn':EPOCAS_CRNN, 'entrada_crnn':[15,6,96],'desfases':DESFASES, 'fusion':{'tipo':'geometrica','dart':.70,'crnn':.30,'piso':1e-7}}

def cargar(manifiesto):
    info = json.loads(manifiesto.read_text('utf8'))
    rutas = {k:manifiesto.parent/v['nombre'] for k,v in info['archivos'].items()}
    if any(sha256(rutas[k])!=v['sha256'] for k,v in info['archivos'].items()):
        raise ValueError('La integridad de los pesos no coincide con el manifiesto.')
    return lgb.Booster(model_file=str(rutas['dart'])),tf.keras.models.load_model(rutas['crnn'],compile=False),info

def guardar(nombre, filas, dart, red, origen, segundos):
    carpeta = STORE/nombre
    carpeta.mkdir(parents=True,exist_ok=True)
    archivos = {}
    for rama,ext in [('dart','.txt'),('crnn','.keras')]:
        temporal = carpeta/(uuid4().hex+ext)
        if rama=='dart':
            dart.save_model(str(temporal))
        else:
            red.save(temporal)
        checksum = sha256(temporal)
        ruta = carpeta/(rama+'-'+checksum[:16]+ext)
        temporal.replace(ruta)
        archivos[rama] = {'nombre':ruta.name,'sha256':checksum}
    spec = contrato(filas)
    info = dict(contrato=spec,firma=huella_json(spec),archivos=archivos,origen=origen,tiempos_s=segundos)
    temporal = carpeta/(uuid4().hex+'.tmp')
    temporal.write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf8')
    temporal.replace(carpeta/'manifest.json')

def ajustar_y_guardar(nombre, filas):
    destino = STORE/nombre/'manifest.json'
    print(f'{nombre} · inicio de DART desde cero', flush=True)
    inicio = perf_counter()
    dart = entrenar_dart(filas)
    intermedio = perf_counter()
    print(f'{nombre} · DART completado en {intermedio-inicio:.1f} s. Inicio de CRNN desde cero.', flush=True)
    red = entrenar_crnn(filas,entradas)
    segundos = {'dart':intermedio-inicio,'crnn':perf_counter()-intermedio}
    guardar(nombre,filas,dart,red,'Ajuste con las personas del manifiesto',segundos)
    print('Ajustado y guardado:',nombre,{k:round(v,1) for k,v in segundos.items()})
    return cargar(destino)

## Entrenamiento con las 100 personas

El ajuste final utiliza todas las noches. Solo después de completar las dos ramas se publican los pesos con sus sumas SHA-256 y el manifiesto en data/models/dart_crnn_mixto/final_100/manifest.json.

In [ ]:
dart_final, crnn_final, ficha_final = ajustar_y_guardar('final_100',meta)
assert ficha_final['contrato']['segmentos_entrenamiento']==len(meta)
display(meta.groupby('cohorte').agg(personas=('subject_id','nunique'),noches=('record_id','nunique'),segmentos=('etapa','size')))
ejemplo = meta.iloc[:10]
rid = ejemplo.record_id.iloc[0]
assert ejemplo.record_id.nunique()==1
p = combinar(dart_final.predict(X.loc[ejemplo.index],num_threads=4),prob_crnn(crnn_final,entradas[rid])[ejemplo.epoch_index])
assert np.isfinite(p).all() and np.allclose(p.sum(1),1)
print('Pesos recargados y fusión verificada.')
print((STORE/'final_100/manifest.json').relative_to(ROOT).as_posix())

display(pd.Series(ficha_final['tiempos_s'], name='Segundos de entrenamiento')
          .rename(index=str.upper).round(1).to_frame())

La verificación de inferencia usa un PSG completo de Telemetry y devuelve una probabilidad por etapa y por segmento. El registro también formó parte del ajuste final. Esta salida verifica la recarga del modelo para inferencia.

In [ ]:
def predecir_edf(archivo, cohorte, dart, crnn, ficha):
    contrato_modelo = ficha['contrato']
    perfil = contrato_modelo['preprocesamiento_dart'][cohorte]
    archivo = Path(archivo)
    huella = sha256(archivo)
    directorio = carpeta_dart_inferencia(perfil, paquete)
    tabla_dart = tabla_dart_guardada(archivo, perfil, directorio, huella)
    features = tabla_dart.reindex(columns=contrato_modelo['columnas_dart']).astype('float32')
    p_tab = dart.predict(features, num_threads=4)
    espectro = entrada_espectral_guardada(archivo, huella, cache_espectral)
    p_red = prob_crnn(crnn, espectro)
    probabilidades = combinar(p_tab, p_red)
    clases = np.array(contrato_modelo['clases'])
    tabla = pd.DataFrame(probabilidades, columns=['p_'+c for c in clases])
    return pd.concat([tabla_dart[['epoch_index','onset_seconds']].reset_index(drop=True),
                      tabla.assign(etapa=clases[probabilidades.argmax(axis=1)])], axis=1)

archivo = registros['Telemetry'][0].psg_path
predicciones = predecir_edf(archivo, 'Telemetry', dart_final, crnn_final, ficha_final)
np.testing.assert_allclose(predicciones.filter(regex='^p_').sum(axis=1), 1, atol=1e-6)
print(archivo.name, '·', len(predicciones), 'segmentos de 30 s')
display(predicciones.head(3).round(4))
print(f'Tiempo total del notebook: {(perf_counter()-inicio_ejecucion)/60:.1f} minutos.')

## Datos para reproducir esta corrida

La evaluación publicada utiliza 197 noches de 100 personas. Las salidas del notebook de evaluación se pueden consultar en GitHub. Para recuperar los EDF, las características y los pesos de esta instantánea desde el remoto DVC del proyecto, comprueba que tu archivo local `data.dvc` contiene este puntero y ejecuta `dvc pull data.dvc` desde la raíz del repositorio. Se necesitan las credenciales de lectura del remoto R2.

```yaml
outs:
- md5: 08de8e50f02834cf42e8a2aea2805917.dir
  size: 21924394614
  nfiles: 1444
  hash: md5
  path: data
```